# Data flow demo

This notebook traces the repository's existing ImmPort-to-GEO pipeline into the files 
consumed by the scientific validator. It is intentionally **read-only**: it inspects 
cached tables and provenance but does not download data or run parsers.

Missing stages are reported and skipped so the notebook can also help diagnose an 
incomplete local run.

In [2]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root")


ROOT = find_repository_root(Path.cwd())
DATA = ROOT / "data"
print(f"Repository root: {ROOT}")

Repository root: /Users/zhouy25/Documents/GitHub/hypothesis2omics


## Pipeline map

```text
ImmPort Tab archives
    │  immport_batch_parse.py
    ▼
Combined sample_manifest.tsv
    │  geo_plan_module.py
    ▼
GEO download plan ──► cached GEO matrices
                         │  geo_matrix_parse_module.py
                         ▼
                 parsed analysis units
                         │
sample manifest + GEO matrix + ImmPort outcomes
                         │  validator_handoff_parse_module.py
                         ▼
                  validator_input/
```

Each transformation has a machine-readable provenance or run-manifest file.

In [3]:
STAGE_FILES = {
    "ImmPort combined manifest": DATA / "immport_cache/parsed/sample_manifest.tsv",
    "ImmPort batch provenance": DATA / "immport_cache/parsed/batch_manifest.json",
    "GEO download plan": DATA / "geo_cache/plan/geo_download_plan.tsv",
    "GEO parse manifest": DATA / "geo_cache/parsed/geo_matrix_parse_manifest.json",
    "Validator bundle manifest": (
        DATA / "validator_input/validator_input_manifest.json"
    ),
}

stage_status = pd.DataFrame(
    [
        {
            "stage artifact": label,
            "available": path.is_file(),
            "size_kib": round(path.stat().st_size / 1024, 1) if path.is_file() else None,
            "repository-relative path": str(path.relative_to(ROOT)),
        }
        for label, path in STAGE_FILES.items()
    ]
)
display(stage_status)

,stage artifact,available,size_kib,repository-relative path
0,ImmPort combined manifest,True,348.0,data/immport_cache/parsed/sample_manifest.tsv
1,ImmPort batch provenance,True,4.0,data/immport_cache/parsed/batch_manifest.json
2,GEO download plan,True,8.0,data/geo_cache/plan/geo_download_plan.tsv
3,GEO parse manifest,True,12.2,data/geo_cache/parsed/geo_matrix_parse_manifest.json
4,Validator bundle manifest,True,3.2,data/validator_input/validator_input_manifest.json


In [4]:
def read_tsv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        print(f"Stage not generated: {path.relative_to(ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)


def read_json(path: Path) -> dict:
    if not path.is_file():
        print(f"Stage not generated: {path.relative_to(ROOT)}")
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(65536), b""):
            digest.update(chunk)
    return digest.hexdigest()

## 1. ImmPort: study design and repository links

The combined sample manifest normalizes study, experiment, subject, biosample, 
timepoint, and public-repository identifiers. The summary below shows how many 
experimental samples and GEO-linked samples each study contributes.

In [5]:
immport_manifest_path = STAGE_FILES["ImmPort combined manifest"]
immport_manifest = read_tsv(immport_manifest_path)

if not immport_manifest.empty:
    immport_summary = (
        immport_manifest.assign(
            geo_linked=immport_manifest["repository_name"].str.casefold().eq("geo")
        )
        .groupby("study_accession", as_index=False)
        .agg(
            experimental_samples=("expsample_accession", "size"),
            subjects=("subject_accession", "nunique"),
            geo_linked_samples=("geo_linked", "sum"),
        )
    )
    display(immport_summary)
    display(immport_manifest.head(3))

,study_accession,experimental_samples,subjects,geo_linked_samples
0,SDY1264,112,25,87
1,SDY1289,302,30,142
2,SDY1291,50,10,50
3,SDY1294,172,21,109
4,SDY1529,252,36,180


,study_accession,experiment_accession,experiment_name,measurement_technique,expsample_accession,result_schema,upload_result_status,repository_name,repository_accession,biosample_accession,...,max_subject_age_in_years,subject_phenotype,subject_location,ancestral_population,ethnicity,gender,race,race_specify,species,strain
0,SDY1264,EXP16736,YF17D-2006 microarray,Transcription profiling by array,ES1191003,PUBLIC REPOSITORY,Parsed,GEO,GSM339820,BS978397,...,45,,US: Georgia,,Not Specified,Not Specified,Not Specified,,Homo sapiens,
1,SDY1264,EXP16736,YF17D-2006 microarray,Transcription profiling by array,ES1191004,PUBLIC REPOSITORY,Parsed,GEO,GSM339821,BS978398,...,45,,US: Georgia,,Not Specified,Not Specified,Not Specified,,Homo sapiens,
2,SDY1264,EXP16736,YF17D-2006 microarray,Transcription profiling by array,ES1191005,PUBLIC REPOSITORY,Parsed,GEO,GSM339822,BS978399,...,45,,US: Georgia,,Not Specified,Not Specified,Not Specified,,Homo sapiens,


## 2. GEO retrieval plan

ImmPort GEO sample accessions are resolved to their parent series and platform. 
The plan keeps download, skip, and unresolved decisions explicit.

In [6]:
geo_plan = read_tsv(STAGE_FILES["GEO download plan"])

if not geo_plan.empty:
    geo_plan["requested_sample_count"] = pd.to_numeric(
        geo_plan["requested_sample_count"], errors="coerce"
    )
    plan_columns = [
        "study_accession",
        "experiment_accession",
        "gse_accession",
        "gpl_accession",
        "download_recommendation",
        "resolution_status",
        "requested_sample_count",
    ]
    display(geo_plan[plan_columns])
    display(
        geo_plan.groupby("download_recommendation", as_index=False)
        .agg(
            plan_rows=("gse_accession", "size"),
            samples=("requested_sample_count", "sum"),
        )
    )

,study_accession,experiment_accession,gse_accession,gpl_accession,download_recommendation,resolution_status,requested_sample_count
0,SDY1264,EXP16736,GSE13485,GPL7567,download,resolved,87
1,SDY1264,EXP16736,GSE13486,GPL7567,skip_superseries,resolved,87
2,SDY1289,EXP15293,GSE13699,GPL6104,download,resolved,126
3,SDY1289,EXP20685,GSE13699,GPL6883,download,resolved,16
4,SDY1294,EXP16733,GSE82152,GPL21975,download,resolved,109
5,SDY1529,EXP28323,GSE125921,GPL10558,download,resolved,36
6,SDY1529,EXP28323,GSE136163,GPL10558,download,resolved,144


,download_recommendation,plan_rows,samples
0,download,6,518
1,skip_superseries,1,87


## 3. Parsed GEO analysis units

Each downloaded study/experiment/series/platform combination becomes one analysis 
unit containing an expression matrix, linked sample rows, GEO metadata, and unit 
provenance.

In [7]:
geo_parse_manifest = read_json(STAGE_FILES["GEO parse manifest"])

if geo_parse_manifest:
    unit_rows = []
    for unit in geo_parse_manifest.get("units", []):
        counts = unit.get("counts", {})
        unit_rows.append(
            {
                "study": unit.get("study_accession"),
                "experiment": unit.get("experiment_accession"),
                "series": unit.get("gse_accession"),
                "platform": unit.get("gpl_accession"),
                "samples": counts.get("samples"),
                "probes": counts.get("probes"),
                "status": unit.get("status"),
            }
        )
    display(pd.DataFrame(unit_rows))
    display(pd.DataFrame([geo_parse_manifest.get("counts", {})]))

,study,experiment,series,platform,samples,probes,status
0,SDY1264,EXP16736,GSE13485,GPL7567,87,20077,success
1,SDY1289,EXP15293,GSE13699,GPL6104,126,22184,success
2,SDY1289,EXP20685,GSE13699,GPL6883,16,24526,success
3,SDY1294,EXP16733,GSE82152,GPL21975,109,18563,success
4,SDY1529,EXP28323,GSE125921,GPL10558,36,47231,success
5,SDY1529,EXP28323,GSE136163,GPL10558,144,47323,success


,download_units,parsed_samples,plan_rows,requested_samples,skipped_plan_rows,units_failed,units_succeeded
0,6,518,7,518,1,0,6


## 4. Scientific-validator input bundle

The bounded handoff copies the normalized sample manifest and extracts only the 
configured GEO feature and ImmPort quantitative outcome. The bundle manifest records 
the exact files, schemas, row counts, and hashes supplied to the validator.

In [8]:
validator_bundle = read_json(STAGE_FILES["Validator bundle manifest"])

if validator_bundle:
    artifact_rows = []
    for name, artifact in validator_bundle.get("artifacts", {}).items():
        artifact_rows.append(
            {
                "artifact": name,
                "rows": artifact.get("rows"),
                "columns": len(artifact.get("columns", [])),
                "size_kib": round(artifact.get("size_bytes", 0) / 1024, 1),
                "sha256": artifact.get("sha256", "")[:12] + "…",
                "path": artifact.get("path"),
            }
        )
    display(pd.DataFrame(artifact_rows))

,artifact,rows,columns,size_kib,sha256,path
0,feature_expression,87,7,5.9,8477be12aaf5…,data/validator_input/feature_expression.tsv
1,quantitative_outcome,25,7,1.8,664dc40b2fc0…,data/validator_input/quantitative_outcome.tsv
2,sample_manifest,888,36,348.0,a42854a6ef46…,data/validator_input/sample_manifest.tsv


## 5. Trace the SDY1264 handoff

This example follows the selected study without performing scientific interpretation. 
GEO sample accessions connect feature values to subjects through the sample manifest; 
ImmPort biosamples provide the quantitative CD8 response for those subjects.

In [ ]:
validator_dir = DATA / "validator_input"
validator_samples = read_tsv(validator_dir / "sample_manifest.tsv")
feature_expression = read_tsv(validator_dir / "feature_expression.tsv")
quantitative_outcome = read_tsv(validator_dir / "quantitative_outcome.tsv")

study = "SDY1264"
required_frames = [validator_samples, feature_expression, quantitative_outcome]
if not any(frame.empty for frame in required_frames):
    study_samples = validator_samples.loc[
        validator_samples["study_accession"].eq(study)
    ].copy()
    study_features = feature_expression.loc[
        feature_expression["study_accession"].eq(study)
    ].copy()
    study_outcomes = quantitative_outcome.loc[
        quantitative_outcome["study_accession"].eq(study)
    ].copy()

    sample_lookup = study_samples.loc[
        study_samples["repository_accession"].ne(""),
        ["repository_accession", "subject_accession", "study_time_collected"],
    ].drop_duplicates("repository_accession")
    feature_links = study_features.merge(
        sample_lookup,
        left_on="sample_accession",
        right_on="repository_accession",
        how="left",
        validate="one_to_one",
    )
    predictor_subjects = set(feature_links["subject_accession"].dropna())
    outcome_subjects = set(study_outcomes["subject_accession"].dropna())
    handoff_summary = pd.DataFrame(
        [
            {
                "study": study,
                "manifest rows": len(study_samples),
                "feature rows": len(study_features),
                "feature-linked subjects": len(predictor_subjects),
                "outcome rows": len(study_outcomes),
                "outcome subjects": len(outcome_subjects),
                "overlapping subjects": len(predictor_subjects & outcome_subjects),
            }
        ]
    )
    display(handoff_summary)
    display(
        feature_links[
            [
                "sample_accession",
                "subject_accession",
                "study_time_collected",
                "gene",
                "expression_value",
            ]
        ].head(5)
    )
    display(
        study_outcomes[
            [
                "biosample_accession",
                "subject_accession",
                "timepoint_day",
                "outcome_name",
                "result_value",
                "result_unit",
            ]
        ].head(5)
    )

## 6. Provenance check

A current checksum can be compared with the checksum recorded when the validator 
bundle was built. A mismatch indicates that an input changed after the recorded run 
and the handoff should be regenerated.

In [ ]:
if validator_bundle:
    checksum_rows = []
    for name, artifact in validator_bundle.get("artifacts", {}).items():
        path = ROOT / artifact["path"]
        current_hash = sha256(path) if path.is_file() else None
        checksum_rows.append(
            {
                "artifact": name,
                "available": path.is_file(),
                "matches recorded hash": current_hash == artifact.get("sha256"),
                "recorded sha256": artifact.get("sha256"),
                "current sha256": current_hash,
            }
        )
    display(pd.DataFrame(checksum_rows))

## Reproduce the handoff

After the upstream cache and parsed manifests exist, recreate the validator bundle 
from the repository root with:

```bash
.venv/bin/python data/validator_handoff_parse_module.py data/validator_handoff_config.json
```

Then rerun this notebook to refresh the displayed counts and provenance checks.